# Swin LCC-FASD — Improved Robust Training (Drive clean data)
This notebook uses the **clean dataset from Google Drive** and trains **two robust strategies only**:
1. **Adversarial Training (PGD-only)**
2. **TRADES (PGD inner solver)**

It starts from **Fine-Tune 3** (`best_3_finetune.pth`) and saves outputs to Drive after every epoch and after final evaluation.

In [ ]:

# ================================================================
# CELL 1 — Install / Imports
# ================================================================
!pip -q install timm

import os, json, math, random, gc, shutil, time
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, confusion_matrix, roc_auc_score
)
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("DEVICE:", DEVICE)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)


DEVICE: cuda


In [ ]:

# ================================================================
# CELL 2 — Mount Google Drive
# ================================================================
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
from pathlib import Path
from google.colab import drive

# 1. Montage du Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Définition du chemin racine à explorer
# On essaie de trouver le dossier peu importe où il est dans ton Drive
def list_directory_contents(target_folder_name):
    print(f"🔎 Recherche du dossier '{target_folder_name}' dans tout le Drive...")
    found_paths = []
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if target_folder_name in dirs:
            found_paths.append(Path(root) / target_folder_name)

    if not found_paths:
        print(f"❌ Dossier '{target_folder_name}' introuvable.")
        return

    for target_path in found_paths:
        print(f"\n✅ DOSSIER TROUVÉ : {target_path}")
        print("-" * 30)

        # Liste des sous-dossiers et fichiers
        items = list(target_path.iterdir())
        if not items:
            print("   (Le dossier est vide)")
        else:
            for item in sorted(items):
                type_item = "📁 [DIR] " if item.is_dir() else "📄 [FILE]"
                print(f"   {type_item} {item.name}")
        print("-" * 30)

# Lancement de l'exploration
list_directory_contents('outputs_swin_lccfasd_robust2')

🔎 Recherche du dossier 'outputs_swin_lccfasd_robust2' dans tout le Drive...

✅ DOSSIER TROUVÉ : /content/drive/MyDrive/outputs_swin_lccfasd_robust2
------------------------------
   📁 [DIR]  3_finetune
   📁 [DIR]  AT_fgsm
   📁 [DIR]  AT_pgd
   📁 [DIR]  HEM
   📁 [DIR]  clean_dataset
   📁 [DIR]  clean_dataset (1)
   📄 [FILE] data_balancing_stats.json
   📁 [DIR]  fgsm_adversarial
   📁 [DIR]  hard_example_mining
   📁 [DIR]  pgd_adversarial
   📁 [DIR]  strategy1A_FGSM
   📁 [DIR]  strategy1_AT
   📁 [DIR]  strategy1_AT_corrected
   📁 [DIR]  strategy2_TRADES
   📁 [DIR]  strategy2_TRADES_corrected
   📁 [DIR]  strategy3_HEM
   📁 [DIR]  strategy3_HEM_corrected
   📁 [DIR]  trades
   📁 [DIR]  trades_at_fgsm
   📁 [DIR]  trades_at_pgd
   📁 [DIR]  trades_mixed_loss
------------------------------


In [ ]:

# ================================================================
# CELL 3 — Paths & Global Configuration
# ================================================================
from pathlib import Path

# DRIVE_ROOT = Path('/content/drive/MyDrive')
# DATA_ROOT  = DRIVE_ROOT / 'clean_dataset'      # expected: train/ val/ test/
# BASE_MODEL_PATH = DRIVE_ROOT / '3_finetune' / 'best_3_finetune.pth'
# OUT_ROOT   = DRIVE_ROOT / 'outputs_swin_lccfasd_robust_improved'
# LOCAL_ROOT = Path('/content/drive/MyDrive/outputs_swin_lccfasd_robust2')


# DRIVE_ROOT = Path('/content/drive/MyDrive')

DRIVE_ROOT = Path('/content/drive/MyDrive')

# C'est ici que se trouvent tes fichiers (le dossier parent)
PROJECT_ROOT = DRIVE_ROOT / 'outputs_swin_lccfasd_robust2'

# --- CHEMINS MIS À JOUR ---
# On pointe à l'intérieur de PROJECT_ROOT
DATA_ROOT       = PROJECT_ROOT / 'clean_dataset'
BASE_MODEL_PATH = PROJECT_ROOT / '3_finetune' / 'best_3_finetune.pth'

# Où enregistrer tes nouveaux résultats d'amélioration
OUT_ROOT        = DRIVE_ROOT / 'outputs_swin_lccfasd_robust_improved'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# OUT_ROOT.mkdir(parents=True, exist_ok=True)
# LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

SPLITS = {
    'train': DATA_ROOT / 'train',
    'val':   DATA_ROOT / 'val',
    'test':  DATA_ROOT / 'test',
}

MODEL_NAME  =  'swin_base_patch4_window7_224'  #'microsoft/swin-base-patch4-window7-224-in22k'
NUM_CLASSES = 2
IMG_SIZE    = 224
MEAN        = [0.485, 0.456, 0.406]
STD         = [0.229, 0.224, 0.225]

GLOBAL_CFG = {
    'batch_size': 16,
    'num_workers': 2,
    'num_epochs': 6,
    'weight_decay': 1e-4,
    'grad_clip': 1.0,
    'label_smoothing': 0.05,
    'early_stop_patience': 3,
    'eps_train': 4/255,
    'pgd_steps_train': 7,
    'pgd_alpha_train': 1/255,
    'trades_beta': 6.0,
    'lr_backbone': 2e-6,
    'lr_classifier': 8e-6,
    'eval_eps_fgsm': 2/255,
    'eval_eps_pgd': 1/255,
    'eval_pgd_steps': 7,
    'eval_pgd_alpha': 1/255,
}

print("="*70)
print("CONFIG")
print("="*70)
print("DATA_ROOT       :", DATA_ROOT)
for k,v in SPLITS.items():
    print(f"{k:<12} : {v} | exists={v.exists()}")
print("BASE_MODEL_PATH :", BASE_MODEL_PATH, "| exists=", BASE_MODEL_PATH.exists())
print("OUT_ROOT        :", OUT_ROOT)
print("-"*70)
for k,v in GLOBAL_CFG.items():
    print(f"{k:<20}: {v}")


CONFIG
DATA_ROOT       : /content/drive/MyDrive/outputs_swin_lccfasd_robust2/clean_dataset
train        : /content/drive/MyDrive/outputs_swin_lccfasd_robust2/clean_dataset/train | exists=True
val          : /content/drive/MyDrive/outputs_swin_lccfasd_robust2/clean_dataset/val | exists=True
test         : /content/drive/MyDrive/outputs_swin_lccfasd_robust2/clean_dataset/test | exists=True
BASE_MODEL_PATH : /content/drive/MyDrive/outputs_swin_lccfasd_robust2/3_finetune/best_3_finetune.pth | exists= True
OUT_ROOT        : /content/drive/MyDrive/outputs_swin_lccfasd_robust_improved
----------------------------------------------------------------------
batch_size          : 16
num_workers         : 2
num_epochs          : 6
weight_decay        : 0.0001
grad_clip           : 1.0
label_smoothing     : 0.05
early_stop_patience : 3
eps_train           : 0.01568627450980392
pgd_steps_train     : 7
pgd_alpha_train     : 0.00392156862745098
trades_beta         : 6.0
lr_backbone         : 2e-06
lr_

In [ ]:

# ================================================================
# CELL 4 — Dataset + Symmetric Augmentation + Balanced Batches
# ================================================================
REAL_NAMES  = {'real'}
SPOOF_NAMES = {'spoof', 'fake', 'attack'}
IMG_EXTS    = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class LCCFASDDataset(Dataset):
    # label convention: 0 = real, 1 = spoof
    def __init__(self, split_dir, transform=None):
        self.transform = transform
        self.samples, self.labels = [], []
        split_dir = Path(split_dir)
        for cls_dir in sorted(split_dir.iterdir()):
            if not cls_dir.is_dir():
                continue
            name = cls_dir.name.lower()
            if name in REAL_NAMES:
                label = 0
            elif name in SPOOF_NAMES:
                label = 1
            else:
                continue
            for f in cls_dir.rglob('*'):
                if f.is_file() and f.suffix.lower() in IMG_EXTS:
                    self.samples.append((f, label))
                    self.labels.append(label)
        if not self.samples:
            raise RuntimeError(f"No images found in {split_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        p, y = self.samples[idx]
        img = Image.open(p).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        return img, y

class BalancedBatchSampler(Sampler):
    def __init__(self, labels, batch_size):
        assert batch_size % 2 == 0, "batch_size must be even"
        self.labels = list(labels)
        self.batch_size = batch_size
        self.half = batch_size // 2
        self.real_idx  = [i for i, y in enumerate(self.labels) if y == 0]
        self.spoof_idx = [i for i, y in enumerate(self.labels) if y == 1]
        if not self.real_idx or not self.spoof_idx:
            raise RuntimeError("Both classes required")
        self.num_batches = math.ceil(max(len(self.real_idx), len(self.spoof_idx)) / self.half)

    def __iter__(self):
        for _ in range(self.num_batches):
            r = np.random.choice(self.real_idx,  size=self.half, replace=True)
            s = np.random.choice(self.spoof_idx, size=self.half, replace=True)
            batch = np.concatenate([r, s])
            np.random.shuffle(batch)
            yield batch.tolist()

    def __len__(self):
        return self.num_batches

def make_loaders():
    train_ds = LCCFASDDataset(SPLITS['train'], train_tf)
    val_ds   = LCCFASDDataset(SPLITS['val'], eval_tf)
    test_ds  = LCCFASDDataset(SPLITS['test'], eval_tf)

    bs, nw = GLOBAL_CFG['batch_size'], GLOBAL_CFG['num_workers']
    train_loader = DataLoader(train_ds, batch_sampler=BalancedBatchSampler(train_ds.labels, bs),
                              num_workers=nw, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=nw, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=bs, shuffle=False, num_workers=nw, pin_memory=True)
    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds

train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = make_loaders()
CLASS_WEIGHTS = torch.tensor([1.0, 1.0], dtype=torch.float32, device=DEVICE)

def show_distribution(*datasets):
    rows = []
    for name, ds in datasets:
        cnt = Counter(ds.labels)
        rows.append({'split': name, 'real': cnt.get(0,0), 'spoof': cnt.get(1,0), 'total': len(ds),
                     'spoof/real': round(cnt.get(1,0)/max(cnt.get(0,1),1), 2)})
    df = pd.DataFrame(rows)
    display(df)

show_distribution(('train', train_ds), ('val', val_ds), ('test', test_ds))
print("Train batches:", len(train_loader), "| Val:", len(val_loader), "| Test:", len(test_loader))

# inspect first 3 train batches
for i, (_, y) in enumerate(train_loader):
    c = Counter(y.tolist())
    print(f"Batch {i+1}: real={c.get(0,0)} spoof={c.get(1,0)}")
    if i == 2:
        break


,split,real,spoof,total,spoof/real
0,train,1173,7035,8208,6.00
1,val,117,1040,1157,8.89
2,test,314,7266,7580,23.14


Train batches: 880 | Val: 73 | Test: 474
Batch 1: real=8 spoof=8
Batch 2: real=8 spoof=8
Batch 3: real=8 spoof=8


In [ ]:

# ================================================================
# CELL 5 — Model / Optimizer / Helpers
# ================================================================
import timm

def build_model():
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)
    return model.to(DEVICE)

def load_base_model():
    assert BASE_MODEL_PATH.exists(), f"Not found: {BASE_MODEL_PATH}"
    model = build_model()
    state = torch.load(BASE_MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(state, strict=False)
    return model

def build_optimizer(model):
    backbone, classifier = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if 'head' in n or 'classifier' in n:
            classifier.append(p)
        else:
            backbone.append(p)
    return torch.optim.AdamW(
        [
            {'params': backbone, 'lr': GLOBAL_CFG['lr_backbone']},
            {'params': classifier, 'lr': GLOBAL_CFG['lr_classifier']},
        ],
        weight_decay=GLOBAL_CFG['weight_decay']
    )

def denorm(x):
    mean = torch.tensor(MEAN, device=x.device).view(1,3,1,1)
    std  = torch.tensor(STD, device=x.device).view(1,3,1,1)
    return x * std + mean

def norm(x):
    mean = torch.tensor(MEAN, device=x.device).view(1,3,1,1)
    std  = torch.tensor(STD, device=x.device).view(1,3,1,1)
    return (x - mean) / std

def clamp_normalized(x):
    x_den = denorm(x)
    x_den = torch.clamp(x_den, 0.0, 1.0)
    return norm(x_den)

def fgsm_attack(model, images, labels, epsilon):
    x = images.detach().clone().requires_grad_(True)
    logits = model(x)
    loss = F.cross_entropy(logits, labels)
    model.zero_grad(set_to_none=True)
    loss.backward()
    adv = x + epsilon * x.grad.sign()
    adv = clamp_normalized(adv.detach())
    return adv

def pgd_attack(model, images, labels, epsilon, alpha, steps, random_start=True):
    ori = images.detach()
    adv = ori.clone()
    if random_start:
        adv = adv + torch.empty_like(adv).uniform_(-epsilon, epsilon)
        adv = clamp_normalized(adv)
    for _ in range(steps):
        adv.requires_grad_(True)
        logits = model(adv)
        loss = F.cross_entropy(logits, labels)
        grad = torch.autograd.grad(loss, adv)[0]
        adv = adv.detach() + alpha * grad.sign()
        delta = torch.clamp(adv - ori, min=-epsilon, max=epsilon)
        adv = clamp_normalized(ori + delta)
    return adv.detach()

def compute_metrics(y_true, y_pred, y_score):
    acc  = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0) * 100
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0) * 100
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0) * 100
    bacc = balanced_accuracy_score(y_true, y_pred) * 100
    cm   = confusion_matrix(y_true, y_pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()  # row: true [real, spoof], col: pred [real, spoof]
    # label convention: 0=real,1=spoof
    # FAR = spoof accepted as real = FN / spoof_total
    # FRR = real rejected as spoof = FP / real_total
    far = (fn / (fn + tp) * 100) if (fn + tp) > 0 else 0.0
    frr = (fp / (tn + fp) * 100) if (tn + fp) > 0 else 0.0
    hter = (far + frr) / 2.0
    auc = roc_auc_score(y_true, y_score) * 100 if len(set(y_true)) == 2 else np.nan
    return {
        'accuracy': acc, 'prec_macro': prec, 'rec_macro': rec, 'f1_macro': f1,
        'bacc': bacc, 'far': far, 'frr': frr, 'hter': hter, 'auc': auc, 'cm': cm.tolist()
    }

@torch.no_grad()
def evaluate(model, loader, criterion=None, attack=None, attack_kwargs=None):
    model.eval()
    attack_kwargs = attack_kwargs or {}
    losses, y_true, y_pred, y_score = [], [], [], []
    for x, y in tqdm(loader, leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        if attack is not None:
            # temporarily enable grads inside attack
            with torch.enable_grad():
                x = attack(model, x, y, **attack_kwargs)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[:,1]
        if criterion is not None:
            losses.append(criterion(logits, y).item())
        preds = logits.argmax(dim=1)
        y_true.extend(y.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        y_score.extend(probs.detach().cpu().numpy().tolist())
    res = compute_metrics(y_true, y_pred, y_score)
    res['loss'] = float(np.mean(losses)) if losses else None
    return res

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2)

def plot_history(history, title, out_dir):
    out_dir = Path(out_dir)
    fig, axes = plt.subplots(1, 3, figsize=(16,4))
    axes[0].plot(history['train_loss'], marker='o'); axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
    axes[1].plot(history['val_f1'], marker='o', color='green'); axes[1].set_title('Val F1 (%)'); axes[1].set_xlabel('Epoch')
    axes[2].plot(history['val_hter'], marker='o', color='red'); axes[2].set_title('Val HTER (%)'); axes[2].set_xlabel('Epoch')
    fig.suptitle(title)
    plt.tight_layout()
    fig.savefig(out_dir / 'training_curves.png', dpi=220, bbox_inches='tight')
    plt.show()
    plt.close(fig)

def plot_cm(cm, title, out_path):
    cm = np.array(cm)
    fig, ax = plt.subplots(figsize=(5.5,5))
    im = ax.imshow(cm)
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['real','spoof']); ax.set_yticklabels(['real','spoof'])
    ax.set_xlabel('Predicted label'); ax.set_ylabel('True label')
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=12)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(out_path, dpi=220, bbox_inches='tight')
    plt.show()
    plt.close(fig)

criterion_eval = nn.CrossEntropyLoss()


In [ ]:

# ================================================================
# CELL 6 — Strategy A: Adversarial Training (PGD-only)
# ================================================================
AT_DIR = OUT_ROOT / 'strategy_AT_PGD_only'
AT_DIR.mkdir(parents=True, exist_ok=True)

def train_one_epoch_at(model, loader, optimizer):
    model.train()
    running_loss, y_true, y_pred = 0.0, [], []
    pbar = tqdm(loader, desc='AT train', leave=False)
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        adv = pgd_attack(
            model, x, y,
            epsilon=GLOBAL_CFG['eps_train'],
            alpha=GLOBAL_CFG['pgd_alpha_train'],
            steps=GLOBAL_CFG['pgd_steps_train'],
            random_start=True
        )
        optimizer.zero_grad(set_to_none=True)
        logits = model(adv)
        loss = F.cross_entropy(logits, y, label_smoothing=GLOBAL_CFG['label_smoothing'])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GLOBAL_CFG['grad_clip'])
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        y_true.extend(y.detach().cpu().numpy().tolist())
        y_pred.extend(preds.detach().cpu().numpy().tolist())
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0) * 100
    return running_loss / len(loader.dataset), f1

print("Loading Fine-Tune 3...")
model_A = load_base_model()
opt_A = build_optimizer(model_A)
sched_A = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_A, mode='max', factor=0.5, patience=1)

best_f1, patience = -1, 0
history_A = {'train_loss': [], 'val_f1': [], 'val_hter': []}

for epoch in range(1, GLOBAL_CFG['num_epochs'] + 1):
    print(f"\n[AT] Epoch {epoch}/{GLOBAL_CFG['num_epochs']}")
    train_loss, train_f1 = train_one_epoch_at(model_A, train_loader, opt_A)
    val_res = evaluate(model_A, val_loader, criterion_eval)
    history_A['train_loss'].append(train_loss)
    history_A['val_f1'].append(val_res['f1_macro'])
    history_A['val_hter'].append(val_res['hter'])
    sched_A.step(val_res['f1_macro'])

    ckpt = {'epoch': epoch, 'model': model_A.state_dict(), 'optimizer': opt_A.state_dict()}
    torch.save(ckpt, AT_DIR / f'checkpoint_epoch_{epoch}.pth')
    save_json(history_A, AT_DIR / 'history.json')
    print({'train_loss': round(train_loss,4), 'train_f1': round(train_f1,2),
           'val_f1': round(val_res['f1_macro'],2), 'val_hter': round(val_res['hter'],2)})

    if val_res['f1_macro'] > best_f1:
        best_f1 = val_res['f1_macro']
        patience = 0
        torch.save(model_A.state_dict(), AT_DIR / 'best_model.pth')
        print("✅ best model updated")
    else:
        patience += 1
        print(f"patience {patience}/{GLOBAL_CFG['early_stop_patience']}")
        if patience >= GLOBAL_CFG['early_stop_patience']:
            print("Early stopping.")
            break

best_A = load_base_model()
best_A.load_state_dict(torch.load(AT_DIR / 'best_model.pth', map_location=DEVICE), strict=False)

test_clean_A = evaluate(best_A, test_loader, criterion_eval)
test_fgsm_A  = evaluate(best_A, test_loader, criterion_eval, attack=fgsm_attack,
                        attack_kwargs={'epsilon': GLOBAL_CFG['eval_eps_fgsm']})
test_pgd_A   = evaluate(best_A, test_loader, criterion_eval, attack=pgd_attack,
                        attack_kwargs={'epsilon': GLOBAL_CFG['eval_eps_pgd'], 'alpha': GLOBAL_CFG['eval_pgd_alpha'],
                                       'steps': GLOBAL_CFG['eval_pgd_steps'], 'random_start': True})

save_json(test_clean_A, AT_DIR / 'test_clean.json')
save_json(test_fgsm_A,  AT_DIR / 'test_fgsm.json')
save_json(test_pgd_A,   AT_DIR / 'test_pgd.json')
plot_history(history_A, 'AT (PGD-only)', AT_DIR)
plot_cm(test_clean_A['cm'], 'AT — Clean', AT_DIR / 'cm_clean.png')
plot_cm(test_fgsm_A['cm'],  'AT — FGSM eval', AT_DIR / 'cm_fgsm.png')
plot_cm(test_pgd_A['cm'],   'AT — PGD eval', AT_DIR / 'cm_pgd.png')

pd.DataFrame([
    {'setting':'clean', **{k:v for k,v in test_clean_A.items() if k!='cm'}},
    {'setting':'fgsm',  **{k:v for k,v in test_fgsm_A.items() if k!='cm'}},
    {'setting':'pgd',   **{k:v for k,v in test_pgd_A.items() if k!='cm'}},
]).to_csv(AT_DIR / 'summary_metrics.csv', index=False)

print("AT done. Saved to:", AT_DIR)


Loading Fine-Tune 3...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]


[AT] Epoch 1/6


AT train:   0%|          | 0/880 [00:00<?, ?it/s]

...

{'train_loss': 0.6222, 'train_f1': 86.72, 'val_f1': 82.24, 'val_hter': np.float64(5.95)}
✅ best model updated

[AT] Epoch 5/6


AT train:   0%|          | 0/880 [00:00<?, ?it/s]

  0%|          | 0/73 [00:00<?, ?it/s]

{'train_loss': 0.5292, 'train_f1': 89.94, 'val_f1': 86.29, 'val_hter': np.float64(4.22)}
✅ best model updated

[AT] Epoch 6/6


AT train:   0%|          | 0/880 [00:00<?, ?it/s]

In [ ]:

# ================================================================
# CELL 7 — Strategy B: TRADES (PGD inner solver)
# ================================================================
TRADES_DIR = OUT_ROOT / 'strategy_TRADES_PGD'
TRADES_DIR.mkdir(parents=True, exist_ok=True)

def trades_loss(model, x, y, beta):
    model.eval()
    x_adv = pgd_attack(
        model, x, y,
        epsilon=GLOBAL_CFG['eps_train'],
        alpha=GLOBAL_CFG['pgd_alpha_train'],
        steps=GLOBAL_CFG['pgd_steps_train'],
        random_start=True
    )
    model.train()
    logits_clean = model(x)
    logits_adv   = model(x_adv)
    ce = F.cross_entropy(logits_clean, y, label_smoothing=GLOBAL_CFG['label_smoothing'])
    kl = F.kl_div(F.log_softmax(logits_adv, dim=1), F.softmax(logits_clean.detach(), dim=1), reduction='batchmean')
    return ce + beta * kl

def train_one_epoch_trades(model, loader, optimizer):
    model.train()
    running_loss, y_true, y_pred = 0.0, [], []
    pbar = tqdm(loader, desc='TRADES train', leave=False)
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = trades_loss(model, x, y, GLOBAL_CFG['trades_beta'])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GLOBAL_CFG['grad_clip'])
        optimizer.step()

        with torch.no_grad():
            logits = model(x)
            preds = logits.argmax(dim=1)

        running_loss += loss.item() * x.size(0)
        y_true.extend(y.detach().cpu().numpy().tolist())
        y_pred.extend(preds.detach().cpu().numpy().tolist())
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0) * 100
    return running_loss / len(loader.dataset), f1

print("Loading Fine-Tune 3...")
model_B = load_base_model()
opt_B = build_optimizer(model_B)
sched_B = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_B, mode='max', factor=0.5, patience=1)

best_f1, patience = -1, 0
history_B = {'train_loss': [], 'val_f1': [], 'val_hter': []}

for epoch in range(1, GLOBAL_CFG['num_epochs'] + 1):
    print(f"\n[TRADES] Epoch {epoch}/{GLOBAL_CFG['num_epochs']}")
    train_loss, train_f1 = train_one_epoch_trades(model_B, train_loader, opt_B)
    val_res = evaluate(model_B, val_loader, criterion_eval)
    history_B['train_loss'].append(train_loss)
    history_B['val_f1'].append(val_res['f1_macro'])
    history_B['val_hter'].append(val_res['hter'])
    sched_B.step(val_res['f1_macro'])

    ckpt = {'epoch': epoch, 'model': model_B.state_dict(), 'optimizer': opt_B.state_dict()}
    torch.save(ckpt, TRADES_DIR / f'checkpoint_epoch_{epoch}.pth')
    save_json(history_B, TRADES_DIR / 'history.json')
    print({'train_loss': round(train_loss,4), 'train_f1': round(train_f1,2),
           'val_f1': round(val_res['f1_macro'],2), 'val_hter': round(val_res['hter'],2)})

    if val_res['f1_macro'] > best_f1:
        best_f1 = val_res['f1_macro']
        patience = 0
        torch.save(model_B.state_dict(), TRADES_DIR / 'best_model.pth')
        print("✅ best model updated")
    else:
        patience += 1
        print(f"patience {patience}/{GLOBAL_CFG['early_stop_patience']}")
        if patience >= GLOBAL_CFG['early_stop_patience']:
            print("Early stopping.")
            break

best_B = load_base_model()
best_B.load_state_dict(torch.load(TRADES_DIR / 'best_model.pth', map_location=DEVICE), strict=False)

test_clean_B = evaluate(best_B, test_loader, criterion_eval)
test_fgsm_B  = evaluate(best_B, test_loader, criterion_eval, attack=fgsm_attack,
                        attack_kwargs={'epsilon': GLOBAL_CFG['eval_eps_fgsm']})
test_pgd_B   = evaluate(best_B, test_loader, criterion_eval, attack=pgd_attack,
                        attack_kwargs={'epsilon': GLOBAL_CFG['eval_eps_pgd'], 'alpha': GLOBAL_CFG['eval_pgd_alpha'],
                                       'steps': GLOBAL_CFG['eval_pgd_steps'], 'random_start': True})

save_json(test_clean_B, TRADES_DIR / 'test_clean.json')
save_json(test_fgsm_B,  TRADES_DIR / 'test_fgsm.json')
save_json(test_pgd_B,   TRADES_DIR / 'test_pgd.json')
plot_history(history_B, 'TRADES (PGD inner)', TRADES_DIR)
plot_cm(test_clean_B['cm'], 'TRADES — Clean', TRADES_DIR / 'cm_clean.png')
plot_cm(test_fgsm_B['cm'],  'TRADES — FGSM eval', TRADES_DIR / 'cm_fgsm.png')
plot_cm(test_pgd_B['cm'],   'TRADES — PGD eval', TRADES_DIR / 'cm_pgd.png')

pd.DataFrame([
    {'setting':'clean', **{k:v for k,v in test_clean_B.items() if k!='cm'}},
    {'setting':'fgsm',  **{k:v for k,v in test_fgsm_B.items() if k!='cm'}},
    {'setting':'pgd',   **{k:v for k,v in test_pgd_B.items() if k!='cm'}},
]).to_csv(TRADES_DIR / 'summary_metrics.csv', index=False)

print("TRADES done. Saved to:", TRADES_DIR)


In [ ]:

# ================================================================
# CELL 8 — Final Comparison
# ================================================================
rows = []
for model_name, res_clean, res_fgsm, res_pgd in [
    ('AT_PGD_only', test_clean_A, test_fgsm_A, test_pgd_A),
    ('TRADES_PGD',  test_clean_B, test_fgsm_B, test_pgd_B),
]:
    for setting, res in [('clean', res_clean), ('fgsm', res_fgsm), ('pgd', res_pgd)]:
        row = {'model': model_name, 'setting': setting}
        row.update({k:v for k,v in res.items() if k != 'cm'})
        rows.append(row)

comp_df = pd.DataFrame(rows)
display(comp_df)
comp_df.to_csv(OUT_ROOT / 'final_comparison.csv', index=False)
save_json(rows, OUT_ROOT / 'final_comparison.json')

for metric in ['f1_macro', 'bacc', 'hter', 'auc']:
    plt.figure(figsize=(7,4))
    for model_name in comp_df['model'].unique():
        sub = comp_df[comp_df['model'] == model_name]
        plt.plot(sub['setting'], sub[metric], marker='o', label=model_name)
    plt.title(f'Comparison — {metric}')
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_ROOT / f'comparison_{metric}.png', dpi=220, bbox_inches='tight')
    plt.show()
    plt.close()

print("Saved final comparison to:", OUT_ROOT)
